In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import logging
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

logging.basicConfig(level=logging.INFO)

In [2]:
# Model Selection
MODEL_ID = "Qwen/Qwen3.5-4B"

In [3]:
# HuggingFace Token status check
HF_TOKEN_STATUS = False
try:
    login()
    HF_TOKEN_STATUS = True
except Exception as e:
    logging.error(f"Huggingface login failed with exception {e}")

logging.info(f"Inspecting LLM Model {MODEL_ID} with HF_TOKEN status {HF_TOKEN_STATUS}")

INFO:root:Inspecting LLM Model Qwen/Qwen3.5-4B with HF_TOKEN status True


In [4]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ---------------------------------------------------------
# 1. THE VOCABULARY & SPECIAL TOKENS
# ---------------------------------------------------------
print("=== Tokenizer Properties ===")
print(f"Vocabulary Size:  {tokenizer.vocab_size:,} tokens")
print(f"All Special Tokens: {tokenizer.all_special_tokens}")
print(f"EOS Token:        '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"PAD Token:        '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

# ---------------------------------------------------------
# ENCODING: Text -> Token IDs
# ---------------------------------------------------------
print("\n=== Text to IDs ===")
sample_text = "Let us inspect Qwen3.5-4b's tokenizer"
print(f"Original Text: '{sample_text}'")

# Encode the text into integer IDs
encoded = tokenizer(sample_text)
input_ids = encoded["input_ids"]
print(f"Token IDs:     {input_ids}")
print(f"Sequence Len:  {len(input_ids)} tokens")

# ---------------------------------------------------------
# DISSECTION: IDs -> Raw String Tokens
# ---------------------------------------------------------
print("\n=== Inspecting the Subwords ===")
# This is the most important debugging command! 
# It shows exactly how the text was sliced.
raw_tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("How the model actually sees the text:")
for idx, (token_id, token_string) in enumerate(zip(input_ids, raw_tokens)):
    # Qwen uses Byte-Pair Encoding (BPE). You will often see 'Ġ' or similar 
    # weird characters which the tokenizer uses to represent spaces.
    print(f"  Token {idx:2}: ID {token_id:<6} -> '{token_string}'")

# ---------------------------------------------------------
# 4. DECODING: IDs -> Text
# ---------------------------------------------------------
print("\n=== IDs back to Text ===")
# Notice how decode() automatically cleans up the weird space characters
decoded = tokenizer.decode(input_ids)
print(f"Decoded: '{decoded}'")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-4B/tr

=== Tokenizer Properties ===
Vocabulary Size:  248,044 tokens
All Special Tokens: ['<|im_end|>', '<|endoftext|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<|image_pad|>', '<|video_pad|>', '<|vision_start|>', '<|vision_end|>']
EOS Token:        '<|im_end|>' (ID: 248046)
PAD Token:        '<|endoftext|>' (ID: 248044)

=== Text to IDs ===
Original Text: 'Let us inspect Qwen3.5-4b's tokenizer'
Token IDs:     [9764, 586, 23331, 1167, 16451, 18, 13, 20, 12, 19, 65, 579, 44424]
Sequence Len:  13 tokens

=== Inspecting the Subwords ===
How the model actually sees the text:
  Token  0: ID 9764   -> 'Let'
  Token  1: ID 586    -> 'Ġus'
  Token  2: ID 23331  -> 'Ġinspect'
  Token  3: ID 1167   -> 'ĠQ'
  Token  4: ID 16451  -> 'wen'
  Token  5: ID 18     -> '3'
  Token  6: ID 13     -> '.'
  Token  7: ID 20     -> '5'
  Token  8: ID 12     -> '-'
  Token  9: ID 19     -> '4'
  Token 10: ID 65     -> 'b'
  Token 11: ID 579    -> ''s'
  Token 12: ID 44424  -> 'Ġtokenizer'

=== IDs back 

In [5]:
# Model loading
# Load the core execution engine (the model graph) into VRAM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    dtype=torch.bfloat16, 
    device_map = "auto" # device_map="auto" # Automatically maps layers to available GPUs
)

# Put the model in evaluation mode (disables dropout, fixes batch norm, etc.)
model.eval()

# This prints the structural representation of the network
print(model)

# Accessing a specific layer deeply nested in the architecture
layer_15 = model.model.layers[15]
print(f"\n\n\n\nLayer 15 attention module: {layer_15.self_attn}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/model.saf

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"


Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2560)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (in_proj_qkv): Linear(in_features=2560, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=2560, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=2560, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=2560, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (down_proj): Linear(

In [6]:
# Inspect raw chat template
print(f"Chat with {MODEL_ID}\nTemplate:\n{tokenizer.chat_template}\n")

Chat with Qwen/Qwen3.5-4B
Template:
{%- set image_count = namespace(value=0) %}
{%- set video_count = namespace(value=0) %}
{%- macro render_content(content, do_vision_count, is_system_content=false) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if 'image' in item or 'image_url' in item or item.type == 'image' %}
                {%- if is_system_content %}
                    {{- raise_exception('System message cannot contain images.') }}
                {%- endif %}
                {%- if do_vision_count %}
                    {%- set image_count.value = image_count.value + 1 %}
                {%- endif %}
                {%- if add_vision_id %}
                    {{- 'Picture ' ~ image_count.value ~ ': ' }}
                {%- endif %}
                {{- '<|vision_start|><|image_pad|><|vision_end|>' }}
            {%- elif 'video' in item or item.type == 'v

In [7]:
# A simple example to illustrate next token prediction
prompt = "I am"

# 1. Encode: String -> Tensor of IDs
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(f"Input shape: {inputs['input_ids'].shape}") 
# Shape: [batch_size=1, sequence_length]

with torch.no_grad(): # Disable gradient tracking to save memory during pure inference
    
    # 2. Forward Pass: Push the tensor through the layer stack
    # By passing output_hidden_states=True, we capture the memory state after every single layer.
    outputs = model(
        **inputs, 
        output_hidden_states=True 
    )
    
    # 3. Analyze the output Logits
    logits = outputs.logits
    print(f"Logits shape: {logits.shape}") 
    # Shape: [batch_size=1, sequence_length, vocab_size]
    
    # We only care about the model's prediction for the *very next* token, 
    # which is the last slice in the sequence dimension.
    next_token_logits = logits[0, -1, :]
    
    # 4. Decode: Find the token ID with the highest probability (Greedy decoding)
    next_token_id = torch.argmax(next_token_logits)
    
    # 5. Detokenize: Integer ID -> String
    next_word = tokenizer.decode(next_token_id)
    print(f"Predicted next token: '{next_word}'")

    # hidden_states[0] is the output of the embedding layer
    # hidden_states[-1] is the output of the final decoder layer
    final_hidden_state = outputs.hidden_states[-1]

    print(f"Hidden state shape: {final_hidden_state.shape}") 
    # Shape: [batch_size=1, sequence_length, hidden_dimension]

    # You can now analyze these vectors mathematically. 
    # For example, extracting the vector representation of the final token:
    last_token_vector = final_hidden_state[0, -1, :]
    print(f"Last token vector: {last_token_vector}") 


Input shape: torch.Size([1, 2])
Logits shape: torch.Size([1, 2, 248320])
Predicted next token: ' trying'
Hidden state shape: torch.Size([1, 2, 2560])
Last token vector: tensor([-0.6992,  3.1719, -3.7812,  ..., -1.4766, -2.0781, -0.1816],
       device='cuda:0', dtype=torch.bfloat16)


In [8]:
# A simple example to illustrate template
# Instantiate streamer, explicitly telling it NOT to print the input prompt
streamer = TextStreamer(tokenizer, skip_prompt=True)
with torch.no_grad(): # Disable gradient tracking to save memory
    print(f"Chat with {MODEL_ID}\nTemplate:\n{tokenizer.chat_template}\n")
    print("-" * 40)
    
    conversation = []
    
    while True:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit"]:
            break
            
        # 1. Update state
        conversation.append({"role": "user", "content": user_input})
        
        # 2. Template & Tokenize (CRITICAL: return_dict=True)
        # This returns a dict with 'input_ids' and 'attention_mask'
        inputs = tokenizer.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            return_dict=True, 
            return_tensors="pt"
        ).to(model.device)

        raw_input_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)
        print("\n[DEBUG] --- Raw Input to Model ---\n")
        print(raw_input_text)
        print("[DEBUG] --------------------------\n")
        print("Assistant: ", end="") # Streamer will append to this line
        
        # 3. Generate
        # Now **inputs safely unpacks input_ids and attention_mask
        output_ids = model.generate(
            **inputs, 
            streamer=streamer, 
            max_new_tokens=1337,
            pad_token_id=tokenizer.eos_token_id # Prevents annoying warning logs
        )

        # 4. Extract only the newly generated tokens
        input_length = inputs["input_ids"].shape[1]
        new_tokens = output_ids[0][input_length:]
        
        # 5. Decode using single decode, not batch_decode
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        # 6. Append to state
        conversation.append({"role": "assistant", "content": response_text})
        print("\n" + "-" * 40)

Chat with Qwen/Qwen3.5-4B
Template:
{%- set image_count = namespace(value=0) %}
{%- set video_count = namespace(value=0) %}
{%- macro render_content(content, do_vision_count, is_system_content=false) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if 'image' in item or 'image_url' in item or item.type == 'image' %}
                {%- if is_system_content %}
                    {{- raise_exception('System message cannot contain images.') }}
                {%- endif %}
                {%- if do_vision_count %}
                    {%- set image_count.value = image_count.value + 1 %}
                {%- endif %}
                {%- if add_vision_id %}
                    {{- 'Picture ' ~ image_count.value ~ ': ' }}
                {%- endif %}
                {{- '<|vision_start|><|image_pad|><|vision_end|>' }}
            {%- elif 'video' in item or item.type == 'v

User:  Who are you



[DEBUG] --- Raw Input to Model ---

<|im_start|>user
Who are you<|im_end|>
<|im_start|>assistant
<think>

[DEBUG] --------------------------

Assistant: Okay, the user is asking "Who are you". I need to respond as Qwen3.5. Let me check the key points from the system instructions. I should mention my name, version, and key capabilities without listing all the technical specs. The user might want a concise introduction. I should avoid mentioning specific parameters like context window or knowledge cutoff unless asked. Focus on being helpful and friendly. Make sure to stay within the guidelines, no sensitive info. Keep it natural and in Chinese since the query is in Chinese. Alright, structure the response to introduce myself clearly and offer assistance.
</think>

I am **Qwen3.5**, the latest large language model developed by Tongyi Lab. I am designed to assist with a wide range of tasks, including answering questions, creating content, logical reasoning, coding, and more. How can I hel

User:  No



[DEBUG] --- Raw Input to Model ---

<|im_start|>user
Who are you<|im_end|>
<|im_start|>assistant
I am **Qwen3.5**, the latest large language model developed by Tongyi Lab. I am designed to assist with a wide range of tasks, including answering questions, creating content, logical reasoning, coding, and more. How can I help you today? 😊<|im_end|>
<|im_start|>user
No<|im_end|>
<|im_start|>assistant
<think>

[DEBUG] --------------------------

Assistant: Okay, the user just said "No" to my previous response. Let me figure out what they mean.

First, I need to recall the conversation history. The user asked "Who are you," and I responded with my identity as Qwen3.5. Now they're saying "No." Maybe they're not satisfied with the answer, or perhaps they want more information. But "No" is pretty vague.

Wait, maybe they're testing if I can handle negative responses. Or maybe they think I'm not the right model. But I am Qwen3.5, the latest from Tongyi Lab. Could they be referring to something 

User:  exit


In [9]:
# A simple example to illustrate tools
def get_database_status(cluster_name: str) -> str:
    """Mock Python function that an agent can execute."""
    print(f"\n[🔧 AGENT EXECUTING TOOL] Checking Python dict for: {cluster_name}...")
    mock_db = {
        "us-east-prod": "ONLINE - 45ms latency",
        "eu-west-staging": "OFFLINE - Error 503"
    }
    return mock_db.get(cluster_name, "Cluster not found.")

# The schema explains the function to the LLM
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_database_status",
            "description": "Check if a database cluster is online.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cluster_name": {
                        "type": "string",
                        "description": "The exact name of the cluster, e.g., 'eu-west-staging'"
                    }
                },
                "required": ["cluster_name"]
            }
        }
    }
]

with torch.no_grad(): # Disable gradient tracking to save memory
    conversation = [
        {"role": "system", "content": "You are a helpful devops assistant."},
        {"role": "user", "content": "Can you check if the eu-west-staging database is healthy?"}
    ]


    inputs = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tools=tools_schema, 
        return_dict=True, 
        return_tensors="pt"
    ).to(model.device)

    raw_input_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)
    print("\n[DEBUG] --- Raw Input to Model ---\n")
    print(raw_input_text)
    print("[DEBUG] --------------------------\n")
    print("Assistant: ", end="") # Streamer will append to this line
    
    output_ids = model.generate(
        **inputs, 
        streamer=streamer, 
        max_new_tokens=1337,
        pad_token_id=tokenizer.eos_token_id # Prevents annoying warning logs
    )

    
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]
    
    # Decode only the newly generated tokens
    raw_output_turn_1 = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print("\n\n\n==========\nNow, your agent can parse the LLM output to call the tool function")
    




[DEBUG] --- Raw Input to Model ---

<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "get_database_status", "description": "Check if a database cluster is online.", "parameters": {"type": "object", "properties": {"cluster_name": {"type": "string", "description": "The exact name of the cluster, e.g., 'eu-west-staging'"}}, "required": ["cluster_name"]}}}
</tools>

If you choose to call a function ONLY reply in the following format with NO suffix:

<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
<parameter=example_parameter_2>
This is the value for the second parameter
that can span
multiple lines
</parameter>
</function>
</tool_call>

<IMPORTANT>
Reminder:
- Function calls MUST follow the specified format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags
- Required parameters MUST be specified
- You may provide optiona